# Phase 6: IMDN XAI pilot

This notebook starts Phase 6 by explaining the best deep-learning model from the comparison stage: IMDN.

The important idea is simple: IMDN produces an image, not a class label. So this notebook explains how parts of the LR input affect the quality of a selected HR output region.

This is a pilot run. It uses a small number of image regions and samples so we can confirm the explanation method before doing a larger run.


## Step 1: Connect Google Drive

The datasets and checkpoints are already in Drive from the earlier phases, so we mount Drive first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2: Prepare the repository and packages

This pulls the latest project code and installs the normal project requirements. It also installs `shap` for the SHAP part of the pilot.

LIME is implemented here using its core idea: change input regions, rerun the model, then fit a small weighted linear explanation. That is easier to control for super-resolution than the usual image-classification LIME wrapper.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap>=0.46,<0.50', 'scikit-learn>=1.5,<1.8'], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and XAI dependencies ready.')


## Step 3: Set paths and pilot settings

A segment means one small block of the LR image. The notebook will hide some segments, run IMDN again, and measure how much the target HR region quality changes.


In [ ]:
import json
import time
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import shap
import torch
from PIL import Image, ImageFilter
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import pairwise_distances
from skimage.metrics import peak_signal_noise_ratio

from app.config import dataset_hr_directory, dataset_lr_directory
from app.deep_learning.alignment import align_reconstruction_to_target
from app.deep_learning.checkpoints import download_official_imdn_checkpoint
from app.deep_learning.imdn import imdn_upsample, load_pretrained_imdn
from app.evaluation.experiment import write_results_csv
from app.evaluation.images import load_rgb_image, validate_hr_lr_dimensions
from app.evaluation.metrics import rgb_to_y

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.')

DEVICE = torch.device('cuda')
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase6' / 'imdn_xai_pilot_v1'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
for directory in (FIGURE_ROOT, METRICS_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

SCALES = (2, 3, 4)
GRID_ROWS = 4
GRID_COLUMNS = 4
LIME_SAMPLES = 40
SHAP_SAMPLES = 40
TARGET_HR_SIZE = 96
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

PILOT_CASES = (
    ('Set5', 'baby.png', 'smooth face and gradual colour regions'),
    ('Set5', 'butterfly.png', 'fine lines and high-frequency texture'),
    ('Set14', 'baboon.png', 'difficult natural texture'),
    ('Urban100', 'img_004.png', 'architectural edges and repetitive structure'),
)

print('GPU:', torch.cuda.get_device_name(DEVICE))
print('Output folder:', OUTPUT_ROOT)


## Step 4: Helper functions

These functions do the repeated work: divide the LR image into blocks, hide selected blocks, run IMDN, score the target HR region, and save heatmaps.


In [ ]:
def make_grid_segments(width, height, rows, columns):
    segments = np.zeros((height, width), dtype=np.int32)
    segment_id = 0
    row_edges = np.linspace(0, height, rows + 1, dtype=int)
    column_edges = np.linspace(0, width, columns + 1, dtype=int)
    for row in range(rows):
        for column in range(columns):
            top, bottom = row_edges[row], row_edges[row + 1]
            left, right = column_edges[column], column_edges[column + 1]
            segments[top:bottom, left:right] = segment_id
            segment_id += 1
    return segments


def centre_target_box(image, size):
    width, height = image.size
    box_width = min(size, width)
    box_height = min(size, height)
    left = (width - box_width) // 2
    top = (height - box_height) // 2
    return (left, top, left + box_width, top + box_height)


def perturb_lr_image(lr_image, segments, keep_mask):
    original = np.asarray(lr_image.convert('RGB'), dtype=np.uint8)
    baseline = np.asarray(lr_image.convert('RGB').filter(ImageFilter.GaussianBlur(radius=3)), dtype=np.uint8)
    hidden = np.isin(segments, np.where(keep_mask == 0)[0])
    output = original.copy()
    output[hidden] = baseline[hidden]
    return Image.fromarray(output, mode='RGB')


def score_target_region(reference_hr, reconstruction, target_box):
    reference_region = np.asarray(reference_hr.crop(target_box).convert('RGB'), dtype=np.float64)
    reconstruction_region = np.asarray(reconstruction.crop(target_box).convert('RGB'), dtype=np.float64)
    reference_y = rgb_to_y(reference_region)
    reconstruction_y = rgb_to_y(reconstruction_region)
    return float(peak_signal_noise_ratio(reference_y, reconstruction_y, data_range=255.0))


def run_imdn_reconstruction(model, lr_image, reference_hr):
    native = imdn_upsample(model, lr_image, DEVICE)
    return align_reconstruction_to_target(native, reference_hr.size).image


def mask_to_score_function(model, reference_hr, lr_image, segments, target_box):
    def score_masks(mask_batch):
        scores = []
        for keep_mask in np.asarray(mask_batch, dtype=int):
            perturbed_lr = perturb_lr_image(lr_image, segments, keep_mask)
            reconstruction = run_imdn_reconstruction(model, perturbed_lr, reference_hr)
            scores.append(score_target_region(reference_hr, reconstruction, target_box))
        return np.asarray(scores, dtype=np.float64)

    return score_masks


def save_importance_figure(lr_image, segments, importance, title, output_path):
    heatmap_lr = np.zeros(segments.shape, dtype=np.float64)
    for segment_id, value in enumerate(importance):
        heatmap_lr[segments == segment_id] = value
    max_abs = max(float(np.max(np.abs(heatmap_lr))), 1e-8)
    heatmap_lr = heatmap_lr / max_abs

    figure, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(lr_image)
    axes[0].set_title('LR input')
    axes[0].axis('off')
    axes[1].imshow(lr_image)
    axes[1].imshow(heatmap_lr, cmap='coolwarm', alpha=0.55, vmin=-1, vmax=1)
    axes[1].set_title(title)
    axes[1].axis('off')
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(figure)
    return output_path


## Step 5: LIME-style explanation

LIME asks: when different input regions are hidden, which regions best explain the score changes? Here the score is PSNR-Y inside the selected HR target region.


In [ ]:
def explain_with_lime(score_masks, feature_count, sample_count):
    masks = rng.integers(0, 2, size=(sample_count, feature_count))
    masks[0, :] = 1
    scores = score_masks(masks)
    full_mask = np.ones((1, feature_count), dtype=int)
    distances = pairwise_distances(masks, full_mask, metric='cosine').reshape(-1)
    kernel_width = 0.75 * np.sqrt(feature_count)
    sample_weights = np.sqrt(np.exp(-(distances ** 2) / (kernel_width ** 2)))
    model = Ridge(alpha=1.0)
    model.fit(masks, scores, sample_weight=sample_weights)
    return model.coef_, float(scores[0]), float(np.mean(scores)), float(np.min(scores))


## Step 6: SHAP explanation

SHAP also changes input regions, but it estimates each region's contribution using a game-theory idea. Because this is expensive, the pilot keeps the number of regions and samples small.


In [ ]:
def explain_with_shap(score_masks, feature_count, sample_count):
    background = np.zeros((1, feature_count), dtype=int)
    full_mask = np.ones((1, feature_count), dtype=int)
    explainer = shap.KernelExplainer(score_masks, background)
    values = explainer.shap_values(full_mask, nsamples=sample_count, silent=True)
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    full_score = float(score_masks(full_mask)[0])
    return values, full_score


## Step 7: Run the pilot

This loops through the selected images and scales. For each case, it loads the LR/HR pair, runs IMDN, then creates one LIME map and one SHAP map.


In [ ]:
records = []
checkpoint_paths = {scale: download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale) for scale in SCALES}

for scale in SCALES:
    model = load_pretrained_imdn(checkpoint_paths[scale], scale, DEVICE)
    for dataset, image_name, region_type in PILOT_CASES:
        hr_path = dataset_hr_directory(dataset, DATA_ROOT) / image_name
        lr_path = dataset_lr_directory(dataset, scale, DATA_ROOT) / image_name
        reference_hr = load_rgb_image(hr_path)
        lr_image = load_rgb_image(lr_path)
        validate_hr_lr_dimensions(reference_hr, lr_image, scale)

        target_box = centre_target_box(reference_hr, TARGET_HR_SIZE)
        segments = make_grid_segments(lr_image.width, lr_image.height, GRID_ROWS, GRID_COLUMNS)
        feature_count = GRID_ROWS * GRID_COLUMNS
        score_masks = mask_to_score_function(model, reference_hr, lr_image, segments, target_box)

        baseline_start = time.perf_counter()
        original_reconstruction = run_imdn_reconstruction(model, lr_image, reference_hr)
        original_psnr_y = score_target_region(reference_hr, original_reconstruction, target_box)
        baseline_seconds = time.perf_counter() - baseline_start

        case_slug = f'{dataset}_{Path(image_name).stem}_x{scale}'
        case_figure_root = FIGURE_ROOT / case_slug
        case_figure_root.mkdir(parents=True, exist_ok=True)

        lime_start = time.perf_counter()
        lime_importance, lime_full_score, lime_mean_score, lime_min_score = explain_with_lime(
            score_masks, feature_count, LIME_SAMPLES
        )
        lime_seconds = time.perf_counter() - lime_start
        lime_path = save_importance_figure(
            lr_image,
            segments,
            lime_importance,
            'LIME importance',
            case_figure_root / 'lime_importance.png',
        )

        shap_start = time.perf_counter()
        shap_importance, shap_full_score = explain_with_shap(score_masks, feature_count, SHAP_SAMPLES)
        shap_seconds = time.perf_counter() - shap_start
        shap_path = save_importance_figure(
            lr_image,
            segments,
            shap_importance,
            'SHAP importance',
            case_figure_root / 'shap_importance.png',
        )

        record = {
            'dataset': dataset,
            'image': image_name,
            'scale': f'x{scale}',
            'region_type': region_type,
            'target_box_left': target_box[0],
            'target_box_top': target_box[1],
            'target_box_right': target_box[2],
            'target_box_bottom': target_box[3],
            'grid_rows': GRID_ROWS,
            'grid_columns': GRID_COLUMNS,
            'segment_count': feature_count,
            'lime_samples': LIME_SAMPLES,
            'shap_samples': SHAP_SAMPLES,
            'original_target_psnr_y': original_psnr_y,
            'lime_full_mask_psnr_y': lime_full_score,
            'lime_mean_perturbed_psnr_y': lime_mean_score,
            'lime_min_perturbed_psnr_y': lime_min_score,
            'shap_full_mask_psnr_y': shap_full_score,
            'baseline_seconds': baseline_seconds,
            'lime_seconds': lime_seconds,
            'shap_seconds': shap_seconds,
            'lime_figure': str(lime_path.relative_to(OUTPUT_ROOT)),
            'shap_figure': str(shap_path.relative_to(OUTPUT_ROOT)),
        }
        records.append(record)
        print(f"PASS: {dataset}/{image_name} x{scale}; target PSNR-Y {original_psnr_y:.4f} dB")
    del model
    torch.cuda.empty_cache()

summary_csv = write_results_csv(records, METRICS_ROOT / 'phase6_xai_pilot_summary.csv', overwrite=True)
print('Saved:', summary_csv)


## Step 8: Save the run settings

This JSON file records what was run, so the results can be repeated later.


In [ ]:
settings = {
    'phase': 6,
    'run_name': 'imdn_xai_pilot_v1',
    'model': 'IMDN',
    'purpose': 'Explain how LR input regions affect selected HR reconstruction quality.',
    'target_metric': 'PSNR-Y on centred HR target region',
    'scales': list(SCALES),
    'pilot_cases': [
        {'dataset': dataset, 'image': image, 'region_type': region_type}
        for dataset, image, region_type in PILOT_CASES
    ],
    'grid_rows': GRID_ROWS,
    'grid_columns': GRID_COLUMNS,
    'lime_samples': LIME_SAMPLES,
    'shap_samples': SHAP_SAMPLES,
    'target_hr_size': TARGET_HR_SIZE,
    'random_seed': RANDOM_SEED,
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
settings_path = METRICS_ROOT / 'phase6_xai_pilot_settings.json'
settings_path.write_text(json.dumps(settings, indent=2) + '\n', encoding='utf-8')
print('Saved:', settings_path)


## Return the results

After running the notebook, return these files from `MyDrive/FYP_SR_Data/results/phase6/imdn_xai_pilot_v1/`:

1. `metrics/phase6_xai_pilot_summary.csv`
2. `metrics/phase6_xai_pilot_settings.json`
3. the `figures/` folder, or a zip of it

Then the written Phase 6 findings can be based on the actual explanation maps, not guesswork.
